# AgriMesh — Crop Health Diagnosis (Layer 07) Training

This notebook unifies the datasets and trains the v1 baseline PyTorch model (MobileNetV3 or EfficientNet) for AgriMesh.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 1. Extract and Unify Datasets
We combine PlantVillage, PlantDoc, and PlantSeg into a single unified dataset.

In [ ]:
import os
import zipfile

DATA_DIR = '/content/dataset'
os.makedirs(DATA_DIR, exist_ok=True)

# Path to your ZIP files in Google Drive
DRIVE_DATASET_DIR = '/content/drive/MyDrive/AgriMesh/datasets'

zips = ['data.zip', 'PlantDoc-Dataset-master.zip', 'PlantSeg-main.zip']

for z in zips:
    zip_path = os.path.join(DRIVE_DATASET_DIR, z)
    if os.path.exists(zip_path):
        print(f"Extracting {z}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(DATA_DIR)
    else:
        print(f"Warning: {zip_path} not found.")


### 2. Prepare PyTorch Dataset & DataLoaders

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

BATCH_SIZE = 32
IMG_SIZE = 224

# Note: You will need to map the raw extracted folders into a clean structure
# For this baseline, we assume the folders are organized as: dataset/tomato_healthy, dataset/tomato_early_blight, etc.

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

try:
    full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform)
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    classes = full_dataset.classes
    print(f"Found {len(classes)} classes: {classes}")
except Exception as e:
    print(f"Error loading dataset: {e}")
    print("Please ensure the dataset is unified into standard PyTorch ImageFolder format first.")


### 3. Initialize Model (MobileNetV3)
We use a lightweight model suitable for edge or cheap server inference.

In [ ]:
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
num_ftrs = model.classifier[3].in_features
model.classifier[3] = nn.Linear(num_ftrs, len(classes) if 'classes' in locals() else 10)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


### 4. Training Loop

In [ ]:
import time

EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    start_time = time.time()
    
    if 'train_loader' not in locals():
        print("Skipping training, dataloaders not ready.")
        break
        
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {running_loss/len(train_loader):.4f} - Time: {time.time()-start_time:.1f}s")
    
    # Validation step
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    print(f"Validation Accuracy: {100 * correct / total:.2f}%")


### 5. Save Model for FastAPI Service

In [ ]:
MODEL_SAVE_PATH = '/content/drive/MyDrive/AgriMesh/models/tomato-v1.pt'
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

torch.save(model, MODEL_SAVE_PATH)
print(f"Model successfully saved to {MODEL_SAVE_PATH}")
print("You can now download this .pt file and place it in your local 'models/' directory for the FastAPI AI Service.")
